# Census Income Data: Cleaning and Exploratory Analysis

This notebook prepares the raw census data for model training. It loads the raw CSV, profiles it to spot quality issues, cleans it (stray spaces in column names and values, duplicate rows), profiles the cleaned result to verify the fixes, and saves it as `data/census_clean.csv`. The training script `train_model.py` consumes that cleaned file.

Note: `data/census.csv` is tracked with DVC, not git. Run `dvc pull` first if the file is missing.

In [ ]:
import pandas as pd
from pandas_profiling import ProfileReport

## Inspecting the raw data

A pandas-profiling report gives a quick statistical overview of every column: value ranges, missing values, and duplicates. Read it before changing anything so the cleaning steps below are driven by what the data actually looks like.

In [ ]:
df = pd.read_csv('data/census.csv')
profile = ProfileReport(df, title="Pandas Profiling Report")
profile

## Cleaning the data

The raw CSV has stray spaces inside column names and values (for example `" education"` and `" Private"`). Spaces like these break column lookups and create duplicate-looking categories, so they are stripped. Duplicate rows are dropped as well, since the census extract should contain one record per person. The index is then reset so it stays a clean 0..N range.

In [ ]:
# Fix space issues in the column names
cols = []
for col in df.columns:
    cols.append(col.replace(' ',''))
df.columns = cols

In [ ]:
# fix space issues in the column values
for col in df.columns: 
    df[col] = df[col].apply(lambda x : str(x).replace(' ',''))
# dropping duplicate rows  
df.drop_duplicates(keep='first', inplace=True)

In [ ]:
df.reset_index(drop=True, inplace=True)

## Verifying the cleaned data and saving it

Profiling the data again confirms the cleaning worked: no stray spaces, no duplicates. The cleaned frame is then saved to `data/census_clean.csv` for the training script to use.

In [ ]:
profile = ProfileReport(df, title="Pandas Profiling Report", minimal=True)

In [ ]:
df.to_csv('data/census_clean.csv')

### Saving the profiling report (optional)

To keep an HTML copy of the report for later review, run:

    profile.to_file("data/cleaned_census.html")

## Checking the saved file

Reloading the saved CSV confirms it round-trips correctly. Note that `to_csv` writes the index as an extra `Unnamed: 0` column, which `train_model.py` drops on load.

In [ ]:
df = pd.read_csv('data/census_clean.csv').drop('Unnamed: 0',axis=1)

## Smoke-testing the prediction API

As a final check, a FastAPI TestClient wraps `src.main` in-process and posts one sample record to `POST /prediction`. This verifies the app module loads, the model and encoder are found, and a request flows end to end.

In [ ]:
from fastapi.testclient import TestClient
import json
from src import main

client = TestClient(main.app)
more_sample = {'age': 50,
 'workclass': 'Private',
 'fnlgt': 154374,
 'education': 'Bachelor',
 'education-num': 13,
 'marital-status': 'Married-civ-spouse',
 'occupation': 'Tech-support',
 'relationship': 'Husband',
 'race': 'White',
 'sex': 'Male',
 'capital-gain': 0,
 'capital-loss': 0,
 'hours-per-week': 40,
 'native-country': 'United-States'}


def test_prediction_bigger():
    r = client.post("/prediction",data=json.dumps(more_sample))
    print(r.json())
    assert r.status_code == 200, r.json()
    assert str(r.json()) == ">50K"

In [ ]:
test_prediction_bigger()

In [ ]:
client.close()
